In [1]:
from src.search import RAGSearch
from langsmith import traceable
from langchain_groq import ChatGroq
from langchain.agents import create_agent

rag_search = RAGSearch() 

model = ChatGroq(model='qwen/qwen3.6-27b',
                 reasoning_effort="none")

agent = create_agent(
    model = model,
    system_prompt='''You are an assistant to a road design engineer. Help the engineer in
                    tackling design problems and other problems related to that field''',
)

## Add decorator
@traceable()
def rag_bot(question:str)->dict:
    ## Relevant context
    query = question
    docs = rag_search.search_and_summarize(query, top_k=10)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       
                        Use the following source documents to answer the user's questions.       
                        If you don't know the answer, just say that you don't know.       
                        Use three sentences maximum and keep the answer concise.

                        Documents:
                        {docs_string}"""
    
    ## llm invoke

    ai_msg=agent.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}

# rag_search = RAGSearch()
# query = 'Give me a summary of the Gradients of an highway'
# summary = rag_search.search_and_summarize(query, top_k=10)
# print('Summary', summary)

c:\Projects\KENYA_ROAD_DESIGN_AGENT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1406.35it/s]


[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Loaded Faiss index and metadata from faiss_store
[INFO] Groq LLM initialized: qwen/qwen3.6-27b


In [ ]:
# ==========================================================================================================================================
# I LEFT AT THE STEP ABOVE

In [3]:
import os
from dotenv import load_dotenv
from pathlib import Path #This is used to give path to the .env to load_dotenv in case it's needed
load_dotenv(override=True)

os.environ['LANGSMITH_API_KEY']=os.getenv('LANGSMITH_API_KEY')
os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['LANGSMITH_TRACING']='true'

In [ ]:
### Create the datapoints
from langsmith import Client

client = Client(
    api_key=os.getenv("LANGSMITH_API_KEY"),
    api_url=os.getenv("LANGSMITH_ENDPOINT"),
)

## Define the dataset - these are your test data

dataset_name = "KeNHA Geometric Design Evaluation"
dataset = client.create_dataset(dataset_name)


examples=[
    {
        "inputs": {"question": "What is the purpose of geometric design?"},
        "outputs": {"answer": "Geometric design establishes the physical layout of a road to provide safe, efficient and economical movement of traffic."}
    },
    {
        "inputs": {"question": "What factors influence the selection of design speed?"},
        "outputs": {"answer": "Design speed depends on terrain, road function, traffic characteristics, safety, and economic considerations."}
    },
    {
        "inputs": {"question": "What are the different terrain classifications used in highway design?"},
        "outputs": {"answer": "Terrain is generally classified as flat, rolling, mountainous, and steep."}
    },
    {
        "inputs": {"question": "Why is stopping sight distance important?"},
        "outputs": {"answer": "Stopping sight distance allows a driver to perceive a hazard and stop safely before reaching it."}
    },
    {
        "inputs": {"question": "What is overtaking sight distance?"},
        "outputs": {"answer": "Overtaking sight distance is the minimum distance required for a vehicle to safely overtake another without conflicting with opposing traffic."}
    },
    {
        "inputs": {"question": "What is the purpose of horizontal curves?"},
        "outputs": {"answer": "Horizontal curves provide a smooth and safe change in road direction."}
    },
    {
        "inputs": {"question": "Why is superelevation provided on horizontal curves?"},
        "outputs": {"answer": "Superelevation counteracts centrifugal force and improves vehicle stability on curves."}
    },
    {
        "inputs": {"question": "What is the purpose of transition curves?"},
        "outputs": {"answer": "Transition curves provide a gradual change from a straight section to a circular curve."}
    },
    {
        "inputs": {"question": "What is a vertical curve?"},
        "outputs": {"answer": "A vertical curve provides a smooth transition between different road gradients."}
    },
    {
        "inputs": {"question": "What factors determine lane width?"},
        "outputs": {"answer": "Lane width depends on design speed, traffic composition, vehicle dimensions, and road classification."}
    }
]


### create the daatset and example in LAngsmith
dataset_name="KeNHA Geometric Design Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)



{'example_ids': ['85ed831a-e95f-449e-a3ee-3b0a53d94e53',
  'e2c4ed06-f5ee-4b68-bfc4-c68d24c3bfd7',
  '59e6d859-f4f5-41ea-ba6d-f063527d01ae',
  '53eba86d-c66a-4e11-aa06-e3f6b9272d9f',
  'a547d80c-3afc-4abc-a147-986031f30fad',
  '694fb89a-cd68-40eb-8641-0e2518e5b232',
  'd3c7cf4d-b64a-4cba-8d54-4b7251bc2aa2',
  '751813c6-5e3b-4c00-851f-e1a2b60f2cce',
  'dcdd99d8-396b-4108-b4b7-497b0327d413',
  '04791537-c171-4329-9702-ae0aa1b167e7'],
 'count': 10,
 'as_of': '2026-06-29T10:56:19.866332396Z'}

In [ ]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

model = ChatGroq(model='qwen/qwen3.6-27b')

agent = create_agent(
    model = model,
    system_prompt="You are an expert professor specialized in grading students' answers to questions",
)

In [ ]:
# import openai
# from langsmith import wrappers

# openai_client =wrappers.wrap_openai(openai.OpenAI())

# eval_instructions = "You are an expert professor specialized in grading students' answers to questions"

def correctness(inputs:dict, outputs:dict,reference_outputs:dict) -> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Response with CORRECT or INCORRECT:
    Grade:
    """
    # response = openai_client.chat.completions.create(
    #     model='gpt-4o-mini',
    #     temperature=0,
    #     messages=[
    #         {'role':'system',
    #          'content': eval_instructions},
    #         {'role': 'user',
    #          'content': user_content}
    #     ]
    # ).choices[0].message.content
    
    response = agent.invoke({'messages': {'role':'user', 'content': user_content}})

    return response['messages'][-1] == 'CORRECT'

In [ ]:
## Concisions - checks whether the actual output is less than 2x the length of the expected result

def concision(outputs: dict, reference_outputs: dict) -> int:
    return int(len(outputs['response']) < 2 * len(reference_outputs['answer']))

In [ ]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

model = ChatGroq(model='qwen/qwen3.6-27b',
                 reasoning_effort="none")

agent1 = create_agent(
    model = model,
    system_prompt="Respond to the users question in a short, concise manner (one short sentence)",
)

In [ ]:
# default_instructions = "Respond to the users question in a short, concise manner (one short sentence)"
def my_app(question: str, 
           #model: str= "qwen/qwen3.6-27b",
             #instructions: str = default_instructions
            ) -> str:
    # return openai_client.chat.completions.create(
    #     model=model,
    #     temperature=0,
    #     messages=[
    #         {"role": "system", "content": instructions},
    #         {"role": "user", "content": question},
    #     ]
    # ).choices[0].message.content
    response = agent1.invoke({'messages': question})

    return response['messages'][-1].content

In [ ]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])} 

In [ ]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="qwen3.6-27b-chatbot"
)